# Customer Support

Here, we show an example of building a customer support chatbot.

This customer support chatbot interacts with SQL database to answer questions.
We will use a mock SQL database to get started: the [Chinook](https://www.sqlitetutorial.net/sqlite-sample-database/) database.
This database is about sales from a music store: what songs and albums exists, customer orders, things like that.

This chatbot has two different states: 
1. Music: the user can inquire about different songs and albums present in the store
2. Account: the user can ask questions about their account

Under the hood, this is handled by two separate agents. 
Each has a specific prompt and tools related to their objective. 
There is also a generic agent who is responsible for routing between these two agents as needed.

In [283]:
!uv sync

Resolved 111 packages in 25ms
Audited 106 packages in 23ms


In [284]:
from dotenv import load_dotenv

load_dotenv()

True

## Load the data

Utils to pull the Chinook database, populate an in-memory SQLite database, and create the engine.

In [285]:
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)

In [286]:
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


## Get sample of every table

In [287]:
import pandas as pd
from sqlalchemy import text, inspect

def sample_all_tables(engine, n=5):
    inspector = inspect(engine)
    tables = inspector.get_table_names()

    samples = {}

    with engine.connect() as conn:
        for table in tables:
            query = text(f"SELECT * FROM {table} LIMIT {n}")
            result = conn.execute(query)

            rows = result.fetchall()
            cols = result.keys()

            df = pd.DataFrame(rows, columns=cols)
            samples[table] = df

            print(f"\n{'='*80}")
            print(f"TABLE: {table} (showing up to {n} rows)")
            print(f"{'='*80}")
            print(df)

    return samples


In [369]:
samples = sample_all_tables(engine, n=5)
print(db.run(
        f"""
        SELECT Track.Name as SongName, Artist.Name as ArtistName 
        FROM Album 
        LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId 
        LEFT JOIN Track ON Track.AlbumId = Album.AlbumId 
        WHERE Artist.Name LIKE 'Aerosmith';
        """))


TABLE: Album (showing up to 5 rows)
   AlbumId                                  Title  ArtistId
0        1  For Those About To Rock We Salute You         1
1        2                      Balls to the Wall         2
2        3                      Restless and Wild         2
3        4                      Let There Be Rock         1
4        5                               Big Ones         3

TABLE: Artist (showing up to 5 rows)
   ArtistId               Name
0         1              AC/DC
1         2             Accept
2         3          Aerosmith
3         4  Alanis Morissette
4         5    Alice In Chains

TABLE: Customer (showing up to 5 rows)
   CustomerId  FirstName     LastName  \
0           1       Luís    Gonçalves   
1           2     Leonie       Köhler   
2           3   François     Tremblay   
3           4      Bjørn       Hansen   
4           5  František  Wichterlová   

                                            Company  \
0  Embraer - Empresa Brasileira de Ae

## Load an LLM

We will load a language model to use.
For this demo we will use OpenAI.

In [290]:
from langchain_openai import ChatOpenAI

# We will set streaming=True so that we can stream tokens
# See the streaming section for more information on this.
model = ChatOpenAI(temperature=0, streaming=True, model="gpt-4o", tags=["router-agent"])

## Load Other Modules

Load other modules we will use.

All of the tools our agents will use will be custom tools. As such, we will use the `@tool` decorator to create custom tools.

We will pass in messages to the agent, so we load `HumanMessage` and `SystemMessage`

In [291]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

## Define the Customer Agent

This agent is responsible for looking up customer information.
It will have a specific prompt as well a specific tool to look up information about that customer (after asking for their user id).

In [292]:
# This tool is given to the agent to look up information about a customer
@tool
def get_customer_info(customer_id: int):
    """Look up customer info given their ID. ALWAYS make sure you have the customer ID before invoking this."""
    return db.run(f"SELECT * FROM Customer WHERE CustomerID = {customer_id};")

In [293]:
customer_prompt = """Your job is to help a user update their profile.

You only have certain tools you can use. These tools require specific input. If you don't know the required input, then ask the user for it.

If you are unable to help the user, you can """

def get_customer_messages(messages):
    return [SystemMessage(content=customer_prompt)] + messages

customer_chain = get_customer_messages | model.bind_tools([get_customer_info]).with_config(tags=["customer-agent"])

## Define the Music Agent

This agent is responsible for figuring out information about music. To do that, we will create a prompt and various tools for looking up information about music

First, we will create indexes for looking up artists and track names.
This will allow us to look up artists and tracks without having to spell their names exactly right.

First, let's create a tool for getting albums by artist.

In [294]:
@tool
def get_albums_by_artist(artist: str):
    """Get albums by an artist."""
    return db.run(
        f"""
        SELECT Album.Title, Artist.Name 
        FROM Album 
        JOIN Artist ON Album.ArtistId = Artist.ArtistId 
        WHERE Artist.Name LIKE '%{artist}%';
        """,
        include_columns=True
    )

Next, lets create a tool for getting tracks by an artist

In [295]:
@tool
def get_tracks_by_artist(artist: str):
    """Get songs by an artist (or similar artists)."""
    return db.run(
        f"""
        SELECT Track.Name as SongName, Artist.Name as ArtistName 
        FROM Album 
        LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId 
        LEFT JOIN Track ON Track.AlbumId = Album.AlbumId 
        WHERE Artist.Name LIKE '%{artist}%';
        """,
        include_columns=True
    )

Finally, let's create a tool for looking up songs by their name.

In [296]:
@tool
def check_for_songs(song_title):
    """Check if a song exists by its name."""
    return db.run(
        f"""
        SELECT * FROM Track WHERE Name LIKE '%{song_title}%';
        """,
        include_columns=True
    )

Create the chain to call the relevant tools

In [297]:
song_system_message = """Your job is to help a customer find any songs they are looking for. 

You only have certain tools you can use. If a customer asks you to look something up that you don't know how, politely tell them what you can help with.

When looking up artists and songs, sometimes the artist/song will not be found. In that case, the tools will return information \
on simliar songs and artists. This is intentional, it is not the tool messing up."""
def get_song_messages(messages):
    return [SystemMessage(content=song_system_message)] + messages

song_recc_chain = get_song_messages | model.bind_tools([get_albums_by_artist, get_tracks_by_artist, check_for_songs]).with_config(tags=["music-agent"])

In [298]:
msgs = [HumanMessage(content="hi! can you help me find songs by amy whinehouse?")]
song_recc_chain.invoke(msgs)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_zytO26kJbl3uNCcI0FLdextW', 'function': {'arguments': '{"artist":"Amy Winehouse"}', 'name': 'get_tracks_by_artist'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_83554c687e', 'service_tier': 'default'}, id='lc_run--b399b41e-87af-4603-87b6-0277bb810720', tool_calls=[{'name': 'get_tracks_by_artist', 'args': {'artist': 'Amy Winehouse'}, 'id': 'call_zytO26kJbl3uNCcI0FLdextW', 'type': 'tool_call'}])

## Define the Generic Agent

We now define a generic agent that is responsible for handling initial inquiries and routing to the right sub agent.

In [299]:
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

class Router(BaseModel):
    """Call this if you are able to route the user to the appropriate representative."""
    choice: str = Field(description="should be one of: music, customer")

system_message = """Your job is to help as a customer service representative for a music store.

You should interact politely with customers to try to figure out how you can help. You can help in a few ways:

- Updating user information: if a customer wants to update the information in the user database. Call the router with `customer`
- Recomending music: if a customer wants to find some music or information about music. Call the router with `music`

If the user is asking or wants to ask about updating or accessing their information, send them to that route.
If the user is asking or wants to ask about music, send them to that route.
Otherwise, respond."""
def get_messages(messages):
    return [SystemMessage(content=system_message)] + messages

In [300]:
chain = get_messages | model.bind_tools([Router])

In [301]:
msgs = [HumanMessage(content="hi! can you help me find a good song?")]
chain.invoke(msgs)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_ujnQhUeYuIPbsGpGnqG8aEWQ', 'function': {'arguments': '{"choice":"music"}', 'name': 'Router'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_83554c687e', 'service_tier': 'default'}, id='lc_run--15c91ae1-43fa-414f-af38-6b0880c8ea10', tool_calls=[{'name': 'Router', 'args': {'choice': 'music'}, 'id': 'call_ujnQhUeYuIPbsGpGnqG8aEWQ', 'type': 'tool_call'}])

In [302]:
msgs = [HumanMessage(content="hi! whats the email you have for me?")]
chain.invoke(msgs)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_RztYe9ywaHKqlUt32fHFF724', 'function': {'arguments': '{"choice":"customer"}', 'name': 'Router'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_83554c687e', 'service_tier': 'default'}, id='lc_run--31a690f0-e21b-4405-a980-da8eaf98ee42', tool_calls=[{'name': 'Router', 'args': {'choice': 'customer'}, 'id': 'call_RztYe9ywaHKqlUt32fHFF724', 'type': 'tool_call'}])

In [303]:
from langchain_core.messages import AIMessage

def add_name(message, name):
    _dict = message.model_dump()
    _dict["name"] = name
    return {"messages": [AIMessage(**_dict)]}

In [304]:
from langgraph.graph import END

def _get_last_ai_message(messages):
    for m in messages[::-1]:
        if isinstance(m, AIMessage):
            return m
    return None


def _is_tool_call(msg):
    return isinstance(msg, AIMessage) and msg.content_blocks[0]["type"] == "tool_call"


def _route(messages):
    last_message = messages["messages"][-1] if messages["messages"] else None
    if isinstance(last_message, AIMessage):
        if not _is_tool_call(last_message):
            return END
        else:
            if last_message.name == "general":
                tool_calls = last_message.content_blocks
                if len(tool_calls) > 1:
                    raise ValueError
                tool_call = tool_calls[0]
                return tool_call['args']['choice']
            else:
                return "tools"
    last_m = _get_last_ai_message(messages["messages"])
    if last_m is None:
        return "general"
    if last_m.name == "music":
        return "music"
    elif last_m.name == "customer":
        return "customer"
    else:
        return "general"

In [305]:
from langgraph.prebuilt import ToolNode

tools = [get_albums_by_artist, get_tracks_by_artist, check_for_songs, get_customer_info]
tools_node = ToolNode(tools)

In [306]:
def _filter_out_routes(messages):
    ms = []
    for m in messages["messages"]:
        if _is_tool_call(m):
            if m.name == "general":
                continue
        ms.append(m)
    return ms

In [307]:
from functools import partial

general_node = _filter_out_routes | chain | partial(add_name, name="general")
music_node = _filter_out_routes | song_recc_chain | partial(add_name, name="music")
customer_node = _filter_out_routes | customer_chain | partial(add_name, name="customer")

In [308]:
from langgraph.graph import MessagesState, StateGraph
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

graph = StateGraph(MessagesState)
nodes = {"general": "general", "music": "music", END: END, "tools": "tools", "customer": "customer"}
# Define a new graph
workflow = StateGraph(MessagesState)
workflow.add_node("general", general_node)
workflow.add_node("music", music_node)
workflow.add_node("customer", customer_node)
workflow.add_node("tools", tools_node)
workflow.add_conditional_edges("general", _route, nodes)
workflow.add_conditional_edges("tools", _route, nodes)
workflow.add_conditional_edges("music", _route, nodes)
workflow.add_conditional_edges("customer", _route, nodes)
workflow.set_conditional_entry_point(_route, nodes)
graph = workflow.compile(checkpointer=memory)

# Evals

## Final Response Evals

Create target function:

In [331]:
import uuid
from langchain_core.messages import HumanMessage

def run_graph(inputs: dict) -> dict:

    thread_id = str(uuid.uuid4())
    result = graph.invoke({"messages": [HumanMessage(content=inputs['Input'])]},config={"configurable": {"thread_id": thread_id}})

    messages = result.get("messages", [])

    if not messages:
        return {"output": ""}

    last_message = messages[-1]

    return {
        "output": last_message.content
    }

Evaluators:

In [334]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT,CONCISENESS_PROMPT

correctness_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="anthropic:claude-sonnet-4-5",
)

conciseness_evaluator = create_llm_as_judge(
    prompt=CONCISENESS_PROMPT,
    feedback_key="conciseness",
    model="anthropic:claude-sonnet-4-5",
)

Run evaluation:

In [ ]:
from langsmith import evaluate

evaluate(
    run_graph,
    data="Chinook support bot: Final Response",
    evaluators=[evaluate_correctness, conciseness_evaluator],
    experiment_prefix="sql-agent-sonnet-final-response"
)

## Single Step Evaluation: Routing

In [339]:
# Target function for running the routing eval
def run_router_eval(inputs: dict) -> dict:
    result = graph.nodes["general"].invoke(inputs["Input"])

    messages = result["messages"]
    if not messages:
        return None

    last_message = messages[-1]
    
    router_tool_calls = [x for x in last_message.tool_calls if x['name'] == 'Router' and 'choice' in x['args']]
    if not router_tool_calls:
        return {
            "choice": None
        }

    return {
        "choice": router_tool_calls[0]['args']['choice']
    }

In [340]:
# Evaluator
def correct_router(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the agent chose the correct route."""
    return outputs["choice"] == reference_outputs['Output']

In [341]:
evaluate(
    run_router_eval,
    data="Chinook support bot: Single Step Routing",
    evaluators=[correct_router],
    experiment_prefix="sql-agent-single-step",
)

View the evaluation results for experiment: 'sql-agent-gpt4o-single-step-7fbd4108' at:
https://smith.langchain.com/o/911e8147-82b3-4451-9946-e96f9331e9f4/datasets/9f0e12ac-c715-4c97-8323-a5a187ae4639/compare?selectedSessions=10f050f4-e267-4ecf-94e1-ec4546b088bb




0it [00:00, ?it/s]

,inputs.Input,outputs.choice,error,reference.Output,feedback.correct_router,execution_time,example_id,id
0,"{'messages': [{'role': 'user', 'content': 'My ...",customer,None,customer,True,1.221285,2d8d1f5c-4106-4ac5-aa04-9d999a90a904,019b00dd-9681-76f3-933c-94f89086fcb4
1,"{'messages': [{'role': 'user', 'content': 'Ref...",None,None,None,True,0.995562,3a6d9889-30b2-48bc-a672-1c8b5981aa32,019b00dd-9b4c-7462-8e7f-c4aa3a93dc2c
2,"{'messages': [{'role': 'user', 'content': 'Wha...",music,None,music,True,0.652774,45add2bb-8b1d-40c4-a584-7648f750ba68,019b00dd-9f33-7239-9283-d1463a80c368
3,"{'messages': [{'role': 'user', 'content': 'I w...",music,None,music,True,0.920744,4b5e93c3-733b-4eac-ac26-9abb9e1bb6b9,019b00dd-a1c4-73b0-844f-df46bca12c1a
4,"{'messages': [{'role': 'user', 'content': 'How...",music,None,music,True,0.564731,51b0bae7-ec55-4f50-8f00-b613bd4a4dd5,019b00dd-a561-744d-93d5-3c7ec1eec30c
5,"{'messages': [{'role': 'user', 'content': 'Fin...",music,None,music,True,0.743658,5be7d67c-cb3c-4d24-bd38-f4adcc45bff8,019b00dd-a798-70ea-a831-f9d393054a57
6,"{'messages': [{'role': 'user', 'content': 'Wha...",None,None,None,True,0.838823,6520d184-08e3-4e6e-994b-06e113a788e0,019b00dd-aa83-7363-83f1-1b787012c83a
7,"{'messages': [{'role': 'user', 'content': 'My ...",customer,None,customer,True,0.475052,655e0d9c-187f-432a-8d0a-9e3f25396898,019b00dd-adce-7283-9ff1-49360934ba48
8,"{'messages': [{'role': 'user', 'content': 'My ...",customer,None,customer,True,0.652977,7ab7ae43-ca7e-42b4-8481-69ffac528ac7,019b00dd-afae-706f-911c-6d2bd86d176b
9,"{'messages': [{'role': 'user', 'content': 'Can...",None,None,None,True,0.753995,819f6c27-df55-4bfe-b184-36f4a503d604,019b00dd-b23f-7500-bb05-4d7e3e7ac1c2


## Trajectory Evaluation: Strict Order

Target function:

In [353]:
import uuid

def run_graph_trajectory(inputs: dict) -> dict:
    """Run graph and track the trajectory it takes along with the final response."""
    trajectory = []

    thread_id = str(uuid.uuid4())

    for chunk in graph.stream({"messages": [HumanMessage(content=inputs['Input'])]}, subgraphs=True, stream_mode="debug", config={"configurable": {"thread_id": thread_id}}):

        if chunk[1]['type'] == 'task':

            trajectory.append(chunk[1]['payload']['name'])

            if chunk[1]['payload']['name'] == 'tools' and chunk[1]['type'] == 'task':
                for tc in chunk[1]['payload']['input']['messages'][-1].tool_calls:
                    trajectory.append(tc['name'])
                    
    return {"trajectory": trajectory}

Evaluators:

In [354]:
def evaluate_extra_steps(outputs: dict, reference_outputs: dict) -> dict:
    """
    Evaluate the number of extra steps in the agent's output.

    Agent performance indicator.
    """
    extra_steps = len(outputs['trajectory']) - len(reference_outputs['Output'])
    return {
        "key": "extra_steps",
        "score": extra_steps,
    }

In [355]:
def evaluate_precision_error_percentage(outputs: dict, reference_outputs: dict) -> dict:
    """
    Precision Error (%): Percent of agent steps that are incorrect vs. the reference (correct order).

    This answers: "How many unneccessary are in the agent's trajectory?"

    It DOES NOT penalize missing required steps (recall).
    """

    i = j = 0
    true_positive = 0

    while i < len(reference_outputs["Output"]) and j < len(outputs["trajectory"]):
        if reference_outputs["Output"][i] == outputs["trajectory"][j]:
            true_positive += 1
            i += 1   # Advance reference only on ordered match
        j += 1       # Always advance through agent output

    total_predicted = len(outputs["trajectory"])
    false_positives = total_predicted - true_positive

    # Percision error = FP / (TP + FP)
    precision_error_pct = (
        (false_positives / total_predicted) * 100 if total_predicted > 0 else 0.0
    )

    return {
        "key": "precision_error_pct",
        "score": precision_error_pct,
    }

In [356]:
def evaluate_recall_error_percentage(outputs: dict, reference_outputs: dict) -> dict:
    """
    Recall Error (%): Percent of required reference steps 
    missing from the agent's trajectory (correct order).
    
    This answers: "How incomplete is the agent's trajectory?"

    It DOES NOT penalize extra or hallucinated steps (precision).
    """
    i = j = 0
    true_positive = 0

    while i < len(reference_outputs["Output"]) and j < len(outputs["trajectory"]):
        if reference_outputs["Output"][i] == outputs["trajectory"][j]:
            true_positive += 1
            i += 1   # Advance reference only on ordered match
        j += 1       # Always advance through agent output

    total_required = len(reference_outputs["Output"])
    false_negatives = total_required - true_positive

    # Recall error = FN / (TP + FN)
    recall_error_pct = (
        (false_negatives / total_required) * 100 if total_required > 0 else 0.0
    )

    return {
        "key": "recall_error_pct",
        "score": recall_error_pct,
    }

Evaluate:

In [358]:
evaluate(
    run_graph_trajectory,
    data="Chinook support bot: Trajectory Strict Match",
    evaluators=[evaluate_extra_steps, evaluate_precision_error_percentage, evaluate_recall_error_percentage],
    experiment_prefix="sql-agent-trajectory",
)

View the evaluation results for experiment: 'sql-agent-trajectory-3936ad02' at:
https://smith.langchain.com/o/911e8147-82b3-4451-9946-e96f9331e9f4/datasets/46328cf5-7266-4910-89e1-d306fcd54d04/compare?selectedSessions=ac5b6eca-7053-4f9a-8a73-8258372ec5eb




0it [00:00, ?it/s]

,inputs.Input,outputs.trajectory,error,reference.Output,feedback.extra_steps,feedback.precision_error_pct,feedback.recall_error_pct,execution_time,example_id,id
0,What is my email address my account id is 4?,"[general, customer, tools, get_customer_info, ...",None,"[general, customer, tools, get_customer_info, ...",0,0.0,0.0,1.765185,1defca93-46df-4cb9-9395-335d190cfc62,019b0105-f511-7102-bab4-4825904b38d5
1,What albums does Aeroosmith have?,"[general, music]",None,"[general, music, tools, get_albums_by_artist, ...",-3,0.0,60.0,1.450263,28df69d6-6e57-445f-b5d6-bbdeec718888,019b0105-fc00-7239-aa91-ec31b2ff6620
2,Refund my last purchase.,[general],None,[general],0,0.0,0.0,0.890267,31dbafa2-056c-404c-acb6-0eaf0b7e30a2,019b0106-01af-735e-a29d-d6099fe288bb
3,"My customer ID is 2, when did I make my last o...","[general, customer, tools, get_customer_info, ...",None,"[general, customer, tools, get_customer_info, ...",0,0.0,0.0,2.105725,406413bb-6f23-4e40-9947-a07528fdd12e,019b0106-052f-77d0-a77e-f71259076d2f
4,What albums did Taylor Swift release?,"[general, music, tools, get_albums_by_artist, ...",None,"[general, music, tools, get_albums_by_artist, ...",0,0.0,0.0,2.768022,4c4e4a0c-fa0b-4752-9a5d-abf5791dfd40,019b0106-0d6f-7109-9dca-761f1d0a2a57
5,"My customer ID is 999999, what is my address?","[general, customer, tools, get_customer_info, ...",None,"[general, customer, tools, get_customer_info, ...",0,0.0,0.0,1.671228,4d8fd75a-67b7-46e3-8cde-80ce8da3c118,019b0106-1849-74fe-8710-37930a9bbadf
6,Can you recommend some songs by Amy Whinehouse?,"[general, music, tools, get_tracks_by_artist, ...",None,"[general, music, tools, get_tracks_by_artist, ...",0,0.0,0.0,1.636478,666fbfc4-99fa-430b-b460-6f9bc8973617,019b0106-1ed8-77f5-a70f-bbd8b1f1ca63
7,Do you have the song Princess of the Dawn?,"[general, music, tools, check_for_songs, music]",None,"[general, music, tools, check_for_songs, music]",0,0.0,0.0,1.933989,7bd2df25-a544-42c7-a104-cd2472ddc848,019b0106-2540-75cf-83b7-d2a29620a731
8,What is my address?,"[general, customer]",None,"[general, customer]",0,0.0,0.0,1.691285,bd96261b-4c5e-488f-bd58-c2de10306b23,019b0106-2cd3-76c0-9566-fa87d5d04680
9,"My customer ID is 2, what is my birthday?","[general, customer, tools, get_customer_info, ...",None,"[general, customer, tools, get_customer_info, ...",0,0.0,0.0,2.384000,d85e5d7a-5caf-4c16-b21f-af685bf8c450,019b0106-3373-77db-99c9-97b4a3d69f12


## Trajectory Evaluation: LLM-as-judge

Target function:

In [365]:
import uuid
def run_graph(inputs: dict) -> dict:

    thread_id = str(uuid.uuid4())
    result = graph.invoke(
        {"messages": [{"role": "user", "content": inputs['Input']}]},
        config={"configurable": {"thread_id": thread_id}},
    )


    extracted = extract_langgraph_trajectory_from_thread(
        graph,
        {"configurable": {"thread_id": thread_id}},
    )

    return {
        "graph_trajectory_inputs": extracted["inputs"],
        "graph_trajectory_outputs": extracted["outputs"],
    }

Evaluator:

In [366]:
from agentevals.graph_trajectory.utils import extract_langgraph_trajectory_from_thread
from agentevals.trajectory.llm import create_trajectory_llm_as_judge


CUSTOM_PROMPT="""
You are an expert data labeler.
Your task is to grade the accuracy of an AI agent's internal trajectory.

<Rubric>
  An accurate trajectory:
  - Makes logical sense between steps
  - Shows clear progression
  - Is relatively efficient, though it does not need to be perfectly efficient
  - Completes the task assigned by the user if all required information is provided
</Rubric>

First, try to understand the goal of the trajectory by looking at the input:

<user_query>
{inputs}
</user_query>

As well as the output of the final message. Once you understand the goal, grade the trajectory
as it relates to achieving that goal.

Grade the following trajectory:

<trajectory>
{outputs}
</trajectory>
"""

graph_trajectory_judge = create_graph_trajectory_llm_as_judge(
    model="anthropic:claude-sonnet-4-5",
    prompt=CUSTOM_PROMPT
)

def evaluate_llm_as_judge(outputs: dict) -> dict:
    """
    LLM-as-judge over the LangGraph trajectory returned by run_graph.
    """
    
    traj_inputs = outputs.get("graph_trajectory_inputs")
    traj_outputs = outputs.get("graph_trajectory_outputs")

    res = graph_trajectory_judge(
        inputs=traj_inputs,
        outputs=traj_outputs,
    )

    return {
        "key": res["key"],         # e.g. "graph_trajectory_accuracy"
        "score": res["score"],     # bool or numeric
        "comment": res.get("comment"),
    }


Run evaluation:

In [367]:
evaluate(
    run_graph,
    data="Chinook support bot: Trajectory LLM as Judge",
    evaluators=[evaluate_llm_as_judge],
    experiment_prefix="sql-agent-sonnet-trajectory_llm_as_judge",
    max_concurrency=4,
)

View the evaluation results for experiment: 'sql-agent-sonnet-trajectory_llm_as_judge-99c1f8a0' at:
https://smith.langchain.com/o/911e8147-82b3-4451-9946-e96f9331e9f4/datasets/5fef4d57-1684-4725-9486-55c6bb312817/compare?selectedSessions=35a640f4-d88f-4766-867f-232f42b02ee6




0it [00:00, ?it/s]

,inputs.Input,outputs.graph_trajectory_inputs,outputs.graph_trajectory_outputs,error,reference.Output,feedback.graph_trajectory_accuracy,execution_time,example_id,id
0,What albums does Aaroosmith have?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, music, tools, get_albums_by_artist, ...",True,1.389718,54741aa6-fb1a-4b2b-8357-d1879e677ac2,019b0118-356e-73d2-ba5c-1a0e6cad10fc
1,"My customer ID is 2, when did I make my last o...","[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, customer, tools, get_customer_info, ...",False,2.998762,034c487d-8cfc-4b22-bca2-c130ee6c1ac5,019b0118-3568-775b-8c3d-785f62d443e7
2,"My customer ID is 2, what is my birthday?","[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, customer, tools, get_customer_info, ...",False,2.653266,8623cf39-1ac6-4cb0-8106-7b61ece2deed,019b0118-3570-72b3-b6fe-329917a70a1c
3,What albums did Taylor Swift release?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, music, tools, get_tracks_by_artist, ...",False,2.018013,949b32d7-64eb-46ef-b02e-52fbc60b2399,019b0118-3adc-75f0-ae88-fe0b76c2a4dc
4,Can you recommend some songs by Amy Whinehouse?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, music, tools, get_tracks_by_artist, ...",True,3.640834,18708aa2-ad54-4209-8aa3-0c608b46ddb4,019b0118-356d-7299-ac55-ff5e5037bf4a
5,Refund my last purchase.,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,[general],False,0.810369,d2f3d305-f3c0-47cf-8f76-a17dc3efb2a1,019b0118-43a7-756e-9f57-adf1ea60e7d4
6,"My customer ID is 999999, what is my address?","[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, customer, tools, get_customer_info, ...",False,1.985252,99acebc8-3487-48ca-811c-24a3f700814d,019b0118-3fcd-7227-9b2f-8f44f60acb84
7,What is my email address my account id is 4?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, customer, tools, get_customer_info, ...",True,1.369989,c849174a-377b-453f-9e3a-0c297c1a3359,019b0118-42c6-7547-ba20-0ec07caa5df8
8,Do you have the song Princess of the Dawn?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, music, tools, check_for_songs, music]",True,1.994910,bdc2d3df-cfff-45a4-953e-f043f407e94a,019b0118-4120-7382-960d-1c33a8164c5e
9,What is my address?,"[{'__start__': {'messages': [{'role': 'user', ...","{'inputs': [], 'results': [{'messages': [{'rol...",None,"[general, customer]",True,1.000277,d32329fb-ca07-4984-9455-a85418cab611,019b0118-46d2-7346-8d6f-f79f943aa578


## Test it out

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import START

history = []
while True:
    user = input('User (q/Q to quit): ')
    if user in {'q', 'Q'}:
        break
    history.append(HumanMessage(content=user))
    async for output in graph.astream({"messages": history}):
        if END in output or START in output:
            continue
        # stream() yields dictionaries with output keyed by node name
        for key, value in output.items():
            print(f"Output from node '{key}':")
            print("---")
            print(value)
        print("\n---\n")
    history.append(AIMessage(content=value["messages"][-1].content))